# 02 — Statistical Testing

**Turning "the chart looks different" into evidence.**

Notebook 01 produced visual impressions. This one tests them. For each factor
we ask a precise question with a precise answer:

| Test | Question it answers |
|---|---|
| One-way ANOVA | Do the Low/Medium/High groups have genuinely different average engagement? |
| Levene's test | Is the ANOVA's equal-variance assumption actually satisfied? |
| Kruskal-Wallis | Does the finding survive without any distribution assumptions? |
| Chi-square | Is this categorical factor independent of performance, or not? |
| Eta² / Cramer's V | The effect is real — but is it *big enough to act on*? |
| Holm-Bonferroni | Having run ~16 tests, which results survive correction for multiple comparisons? |

The last two rows are what separate this from a typical submission. A p-value
alone tells you an effect exists; it says nothing about whether the effect is
large enough to justify spending a school's intervention budget.

In [ ]:
# Make `src` importable no matter where Jupyter was launched from.
import sys, warnings
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config" / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

print(f"Project root: {ROOT}")

In [ ]:
from src.data.preprocess import load_processed
from src.analysis.statistical_tests import run_all_tests, summarise_findings, SCIPY_AVAILABLE
from src.utils.config import load_config

cfg = load_config()
df = load_processed(cfg)
print(f"Students: {len(df)}   |   SciPy backend available: {SCIPY_AVAILABLE}")

## 1. Run the full test battery

One call runs every ANOVA, every chi-square, both assumption checks, and the
multiple-comparison correction across the whole family at once.

In [ ]:
results = run_all_tests(df, cfg)
print(summarise_findings(results, top_n=8))

## 2. ANOVA results in detail

`F` is the ratio of between-group variance to within-group variance —
"how far apart are the group averages, relative to how noisy each group is
internally?"

`eta²` is the share of the feature's total variance explained by performance
band. Conventional bands: 0.01 small, 0.06 medium, 0.14 large.

In [ ]:
import pandas as pd

anova_rows = []
for feature, r in results["anova"].items():
    anova_rows.append({
        "Factor": r["friendly_name"],
        "F": r["f_statistic"],
        "p (raw)": r["p_value"],
        "p (Holm)": r["p_value_adjusted"],
        "eta²": r["eta_squared"],
        "Effect": r["effect_size_label"],
        "Mean L": r["group_means"]["L"],
        "Mean M": r["group_means"]["M"],
        "Mean H": r["group_means"]["H"],
        "Significant": r["significant"],
    })

pd.DataFrame(anova_rows).sort_values("eta²", ascending=False).reset_index(drop=True)

### Assumption checks — being honest about ANOVA

ANOVA assumes the three groups have roughly equal variance. Levene's test
checks that. Where the assumption fails, the ANOVA p-value on its own is not
fully trustworthy — so we report Kruskal-Wallis, which makes no such
assumption, right next to it.

**When both agree, the conclusion is safe no matter which assumptions you are
willing to make.** That agreement column is the one that matters.

In [ ]:
assumption_rows = []
for feature, r in results["anova"].items():
    assumption_rows.append({
        "Factor": r["friendly_name"],
        "Levene W": r["levene_w"],
        "Levene p": round(r["levene_p"], 4),
        "Equal variance OK?": r["equal_variance_assumption_holds"],
        "Kruskal H": r["kruskal_h"],
        "Kruskal p": r["kruskal_p"],
        "Both tests agree?": r["parametric_and_nonparametric_agree"],
    })

pd.DataFrame(assumption_rows)

## 3. Chi-square results

For categorical factors the null hypothesis is independence: "knowing this
feature tells you nothing about the performance band."

`min_expected` is the assumption check. The standard rule is that no expected
cell count should drop below 5 — where it does, the chi-square approximation
becomes unreliable and we flag it rather than quietly reporting the number.

In [ ]:
chi_rows = []
for feature, r in results["chi_square"].items():
    chi_rows.append({
        "Factor": r["friendly_name"],
        "chi²": r["chi2"],
        "dof": r["dof"],
        "p (raw)": r["p_value"],
        "p (Holm)": r["p_value_adjusted"],
        "Cramer's V": r["cramers_v"],
        "Effect": r["effect_size_label"],
        "Min expected": r["min_expected"],
        "Assumption OK?": r["assumption_ok"],
        "Significant": r["significant"],
    })

pd.DataFrame(chi_rows).sort_values("Cramer's V", ascending=False).reset_index(drop=True)

## 4. Why we corrected for multiple comparisons

We ran 16 hypothesis tests. If every null hypothesis were true, the chance of
at least one "significant" result purely by luck is:

$$1 - (1 - 0.05)^{16} \approx 56\%$$

More likely than a coin flip. Holm-Bonferroni controls that family-wise error
rate while rejecting more nulls than plain Bonferroni would, so it costs less
real signal. The cell below shows what the correction actually changed.

In [ ]:
n = results["n_tests_in_family"]
print(f"Tests run: {n}")
print(f"P(at least one false positive without correction): {1 - 0.95 ** n:.1%}\n")

changed = [
    r for r in results["ranked_factors"]
    if (r["p_value"] < 0.05) != (r["p_value_adjusted"] < 0.05)
]
if changed:
    print("Results that changed verdict after correction:")
    for r in changed:
        print(f"  {r['friendly_name']}: raw p={r['p_value']:.4f} -> Holm p={r['p_value_adjusted']:.4f}")
else:
    print("No result changed verdict — every finding survives correction.")

## 5. The ranking that drives the whole project

Everything downstream — which features the recommendation engine talks about, which levers the cohort simulator exposes — traces back to this ordering.

In [ ]:
from src.analysis.eda import apply_house_style, plot_effect_sizes
from IPython.display import Image, display

apply_house_style()
p, cap = plot_effect_sizes(results, cfg)
display(Image(str(p)))
print(cap)

## 6. Checkpoint summary

Write-up for the report, generated from the numbers above rather than typed
by hand.

In [ ]:
print(summarise_findings(results, top_n=6))
print()
print("Factors with NO significant relationship to performance:")
for r in results["ranked_factors"]:
    if not r["significant"]:
        print(f"  - {r['friendly_name']}: p={r['p_value']:.3f} (Holm-adjusted {r['p_value_adjusted']:.3f})")

### What this means in practice

The significant, actionable factors are attendance, resource use, hands raised
and announcements read. Those four are what a school can realistically move.

Nationality, place of birth and gender are also statistically significant —
but they are **not levers**. They are demographic attributes, and a system that
"recommended" changing them would be both useless and offensive. Their
significance here is precisely the reason Phase 6 runs a fairness audit: if
these attributes correlate with the outcome, we have to check whether the model
is quietly using them to make decisions.